# Signal Types Classification (ЧЕРНОВИК)

## Цель проекта

Целью проекта является автоматическая кластеризация сигналов, полученных со сцинтилляционного детектора, на три группы.

Это ЧЕРНОВИК с разнми вариантами предобработки, наборами признаков, PCA, моделями кластеризации и перенумерации кластеров, которые проверялись по ходу работы.

Начальные эксперименты дали не очень хороший результат на Kaggle: на этом этапе лучший полученный score не поднимался выше 0.46854. Ниже оставлены промежуточные шаги и варианты, от которых пришлось отказаться.

## Импорт библиотек

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from scipy.signal import find_peaks
from scipy.integrate import trapezoid

from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, MiniBatchKMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [ ]:
from google.colab import drive # подключение Google Drive
drive.mount('/content/drive')

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Signal_types_classification"
DATA_PATH = os.path.join(PROJECT_DIR, "Run200_Wave_0_1.txt")

print("Папка проекта:", PROJECT_DIR)
print("Файл данных:", DATA_PATH)
print("Файл найден:", os.path.exists(DATA_PATH))

## Работа с данными
### Загрузка данных

In [ ]:
raw_test = pd.read_csv(
    DATA_PATH,
    sep=" ",
    header=None,
    skipinitialspace=True
)

print("Размер до удаления столбцов:", raw_test.shape)

dataset = raw_test.drop([0, 1, 2, 3, 504], axis=1)
dataset.columns = list(range(500))

signals = dataset.copy()

print("Размер signals:", signals.shape)
signals.head()

### Структура данных

Каждая строка соответствует одному зарегистрированному сигналу.

Первые несколько столбцов являются служебными параметрами записи, а последующие столбцы содержат временной ряд сигнала (значения амплитуды во времени).

В дальнейшем первые 4 столбца будем рассматривать как метаданные, а остальные 500 столбцов как сам сигнал.

### Разделение метаданных и сигналов

In [ ]:
# Метаданные — первые 4 служебных столбца из исходного файла
meta = raw_test.iloc[:, :4].copy()

# Сигналы уже были получены строго по примеру из test-dataset.ipynb
# dataset = raw_test.drop([0, 1, 2, 3, 504], axis=1)
# dataset.columns = list(range(500))
signals = dataset.copy()

print("Размер meta:", meta.shape)
print("Размер signals:", signals.shape)

signals.head()

In [ ]:
print("Количество сигналов:", signals.shape[0])
print("Длина одного сигнала:", signals.shape[1])

signals.describe().T.head()

### Проверка пропусков

In [ ]:
print("Пропуски в signals:", signals.isna().sum().sum())
print("Пропуски в meta:", meta.isna().sum().sum())

### Визуализация первых сигналов в исходном виде

In [ ]:
n_examples = 100

ax = signals.iloc[:n_examples].T.plot(
    legend=False,
    figsize=(16, 6),
    title="Первые 100 сигналов в исходном виде"
)

ax.set_xlabel("Время, отсчёты")
ax.set_ylabel("ADC")
plt.show()

### Преобразование сигнала

In [ ]:
processed_signals = 2**14 - signals - 1560

print("Размер processed_signals:", processed_signals.shape)
processed_signals.head()

In [ ]:
N = range(0, 100)

ax = processed_signals.T[N][140:200].plot(
    title="Signal after transformation",
    legend=None,
    figsize=(20, 10)
)

ax.set_xlabel("time, ns")
ax.set_ylabel("bit ADC")
plt.show()

### Визуализация преобразованных сигналов

In [ ]:
ax = processed_signals.iloc[:n_examples].T.plot(
    legend=False,
    figsize=(16, 6),
    title="Первые 100 сигналов после преобразования"
)

ax.set_xlabel("Время, отсчёты")
ax.set_ylabel("Преобразованная амплитуда")
plt.show()

### Увеличенный участок импульса

In [ ]:
ax = processed_signals.iloc[:n_examples, 120:220].T.plot(
    legend=False,
    figsize=(16, 6),
    title="Область основного импульса"
)

ax.set_xlabel("Время, отсчёты")
ax.set_ylabel("Преобразованная амплитуда")
plt.show()

## Первичные наблюдения

После визуализации видно, что основная часть импульса расположена примерно в одной временной области. Это позволяет извлекать физически интерпретируемые признаки:

- максимальная амплитуда,
- время достижения максимума,
- площадь под сигналом,
- площадь в раннем и позднем временных окнах,
- отношение хвостовой части сигнала к полной площади,
- ширина импульса,
- статистические характеристики формы сигнала.

Разные типы частиц могут иметь похожую амплитуду, но отличаться формой импульса и долей медленной хвостовой компоненты, поэтому эти признаки важны для анализа.

### Функция генерации признаков

In [ ]:
def extract_signal_features(signal_df):
    """
    Извлекает признаки из временных рядов сигналов.
    На вход: signal_df: DataFrame размера (n_objects, n_time_points)
    На выход: DataFrame с признаками для кластеризации
    """

    X = signal_df.values.astype(float)
    n_objects, n_points = X.shape

    features = pd.DataFrame(index=signal_df.index)

    # Базовые характеристики
    features["max_amp"] = X.max(axis=1)
    features["min_amp"] = X.min(axis=1)
    features["mean_amp"] = X.mean(axis=1)
    features["std_amp"] = X.std(axis=1)
    features["median_amp"] = np.median(X, axis=1)

    # Положение максимума
    peak_idx = X.argmax(axis=1)
    features["peak_idx"] = peak_idx

    # Площадь под сигналом
    X_positive = np.clip(X, 0, None)
    features["area_total"] = X_positive.sum(axis=1)

    # Окна вокруг импульса
    windows = {
        "pre": (0, 120),
        "rise": (120, 150),
        "peak": (140, 180),
        "early": (150, 220),
        "middle": (220, 320),
        "tail": (320, n_points)
    }

    for name, (start, end) in windows.items():
        start = max(0, start)
        end = min(n_points, end)

        area = X_positive[:, start:end].sum(axis=1)

        features[f"area_{name}"] = area
        features[f"ratio_{name}"] = area / (features["area_total"] + 1e-9)

    # Отношения хвоста к другим частям сигнала
    features["tail_to_peak"] = features["area_tail"] / (features["area_peak"] + 1e-9)
    features["tail_to_early"] = features["area_tail"] / (features["area_early"] + 1e-9)
    features["middle_to_peak"] = features["area_middle"] / (features["area_peak"] + 1e-9)

    # Квантили
    features["q05"] = np.quantile(X, 0.05, axis=1)
    features["q25"] = np.quantile(X, 0.25, axis=1)
    features["q75"] = np.quantile(X, 0.75, axis=1)
    features["q95"] = np.quantile(X, 0.95, axis=1)

    # Статистики формы
    features["skew"] = stats.skew(X, axis=1)
    features["kurtosis"] = stats.kurtosis(X, axis=1)

    # Производная сигнала
    dX = np.diff(X, axis=1)
    features["diff_max"] = dX.max(axis=1)
    features["diff_min"] = dX.min(axis=1)
    features["diff_std"] = dX.std(axis=1)

    # Ширина на половине максимума
    widths = []

    for i in range(n_objects):
        row = X[i]
        half_max = row.max() * 0.5
        above = np.where(row >= half_max)[0]

        if len(above) > 0:
            width = above[-1] - above[0]
        else:
            width = 0

        widths.append(width)

    features["width_half_max"] = widths

    # Логарифмы масштабных признаков
    for col in [
        "max_amp",
        "area_total",
        "area_peak",
        "area_early",
        "area_middle",
        "area_tail",
        "std_amp"
    ]:
        features[f"log_{col}"] = np.log1p(np.maximum(features[col], 0))

    # Очистка возможных inf/nan
    features = features.replace([np.inf, -np.inf], np.nan)
    features = features.fillna(features.median(numeric_only=True))

    return features

### Извлечение признаков

In [ ]:
features = extract_signal_features(processed_signals)

print("Размер таблицы признаков:", features.shape)
features.head()

### Анализ признаков

In [ ]:
features.describe().T.sort_values("std", ascending=False).head(20)

### Распределение важных признаков

In [ ]:
important_features = [
    "max_amp",
    "area_total",
    "ratio_tail",
    "tail_to_peak",
    "peak_idx",
    "width_half_max"
]

for col in important_features:
    plt.figure(figsize=(10, 4))
    plt.hist(features[col], bins=80)
    plt.title(f"Распределение признака: {col}")
    plt.xlabel(col)
    plt.ylabel("Количество сигналов")
    plt.show()

### Матрица корреляций

In [ ]:
corr_features = [
    "max_amp",
    "area_total",
    "ratio_peak",
    "ratio_early",
    "ratio_middle",
    "ratio_tail",
    "tail_to_peak",
    "tail_to_early",
    "peak_idx",
    "width_half_max",
    "skew",
    "kurtosis"
]

corr_matrix = features[corr_features].corr()

plt.figure(figsize=(12, 10))
plt.imshow(corr_matrix, aspect="auto")
plt.colorbar()
plt.xticks(range(len(corr_features)), corr_features, rotation=90)
plt.yticks(range(len(corr_features)), corr_features)
plt.title("Корреляция основных признаков")
plt.show()

### Подготовка признаков для моделей

In [ ]:
# Не все признаки одинаково полезны
# Выберем признаки, связанные с формой, амплитудой и временной структурой сигнала

selected_features = [
    "log_max_amp",
    "log_area_total",
    "log_area_peak",
    "log_area_early",
    "log_area_middle",
    "log_area_tail",
    "ratio_peak",
    "ratio_early",
    "ratio_middle",
    "ratio_tail",
    "tail_to_peak",
    "tail_to_early",
    "middle_to_peak",
    "peak_idx",
    "width_half_max",
    "skew",
    "kurtosis",
    "diff_max",
    "diff_min",
    "diff_std"
]

X_features = features[selected_features].copy()

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X_features)

print("Размер матрицы для обучения:", X_scaled.shape)

### PCA для визуализации и снижения размерности

In [ ]:
pca = PCA(n_components=10, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_

print("Доля объяснённой дисперсии по компонентам:")
for i, value in enumerate(explained, start=1):
    print(f"PC{i}: {value:.4f}")

print("Суммарно:", explained.sum().round(4))

### График объяснённой дисперсии

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(explained) + 1), explained, marker="o")
plt.title("Доля объяснённой дисперсии PCA")
plt.xlabel("Номер компоненты")
plt.ylabel("Explained variance ratio")
plt.show()

### Функция оценки кластеризации

In [ ]:
# Считает внутренние метрики качества кластеризации

def evaluate_clustering(X, labels, model_name):

    unique_labels = np.unique(labels)

    if len(unique_labels) < 2:
        return {
            "model": model_name,
            "n_clusters": len(unique_labels),
            "silhouette": np.nan,
            "calinski_harabasz": np.nan,
            "davies_bouldin": np.nan
        }

    return {
        "model": model_name,
        "n_clusters": len(unique_labels),
        "silhouette": silhouette_score(X, labels),
        "calinski_harabasz": calinski_harabasz_score(X, labels),
        "davies_bouldin": davies_bouldin_score(X, labels)
    }

### Обучение моделей кластеризации

In [ ]:
# Для кластеризации используем первые несколько PCA-компонент
# Это снижает шум и ускоряет обучение

X_model = X_pca[:, :6]

models = {}

models["KMeans"] = KMeans(
    n_clusters=3,
    n_init=50,
    random_state=RANDOM_STATE
)

models["MiniBatchKMeans"] = MiniBatchKMeans(
    n_clusters=3,
    batch_size=2048,
    n_init=50,
    random_state=RANDOM_STATE
)

models["GaussianMixture"] = GaussianMixture(
    n_components=3,
    covariance_type="full",
    n_init=10,
    random_state=RANDOM_STATE
)

labels_dict = {}
scores = []

for name, model in models.items():
    if name == "GaussianMixture":
        labels = model.fit_predict(X_model)
    else:
        labels = model.fit_predict(X_model)

    labels_dict[name] = labels
    scores.append(evaluate_clustering(X_model, labels, name))

scores_df = pd.DataFrame(scores)
scores_df

## Выбор лучшей модели

In [ ]:
# Для silhouette больше - лучше
# Для Calinski-Harabasz больше - лучше
# Для Davies-Bouldin меньше - лучше

scores_df_sorted = scores_df.sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True]
)

scores_df_sorted

In [ ]:
best_model_name = scores_df_sorted.iloc[0]["model"]
best_labels_raw = labels_dict[best_model_name]

print("Лучшая модель по внутренним метрикам:", best_model_name)
print("Размеры кластеров:")
print(pd.Series(best_labels_raw).value_counts().sort_index())

### Визуализация кластеров в PCA

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=best_labels_raw,
    s=8,
    alpha=0.7
)
plt.title(f"Кластеры модели {best_model_name} в пространстве PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="cluster")
plt.show()

### Анализ полученных кластеров

In [ ]:
cluster_analysis = features.copy()
cluster_analysis["cluster_raw"] = best_labels_raw

summary = cluster_analysis.groupby("cluster_raw")[[
    "max_amp",
    "area_total",
    "ratio_peak",
    "ratio_early",
    "ratio_middle",
    "ratio_tail",
    "tail_to_peak",
    "tail_to_early",
    "peak_idx",
    "width_half_max",
    "skew",
    "kurtosis"
]].median()

summary["count"] = cluster_analysis.groupby("cluster_raw").size()

summary

### Средняя форма сигнала по кластерам

In [ ]:
plt.figure(figsize=(14, 6))

for cluster in sorted(np.unique(best_labels_raw)):
    idx = np.where(best_labels_raw == cluster)[0]
    mean_signal = processed_signals.iloc[idx].mean(axis=0)
    plt.plot(mean_signal.values, label=f"cluster {cluster}, n={len(idx)}")

plt.title("Средняя форма сигнала по кластерам")
plt.xlabel("Время, отсчёты")
plt.ylabel("Преобразованная амплитуда")
plt.legend()
plt.show()

### Сигналы внутри каждого кластера

In [ ]:
for cluster in sorted(np.unique(best_labels_raw)):
    idx = np.where(best_labels_raw == cluster)[0][:50]

    plt.figure(figsize=(14, 5))
    plt.plot(processed_signals.iloc[idx].T, alpha=0.25)
    plt.title(f"Примеры сигналов из кластера {cluster}")
    plt.xlabel("Время, отсчёты")
    plt.ylabel("Преобразованная амплитуда")
    plt.show()

## Интерпретация кластеров

После кластеризации были получены три группы сигналов. Для интерпретации использовались средняя форма сигнала в каждом кластере, амплитуда, площадь под сигналом, доля поздней хвостовой части, положение максимума, ширина импульса.

Кластеры с разной долей хвостовой компоненты могут соответствовать разным типам частиц, т. к. форма импульса отражает особенности взаимодействия частицы со сцинтиллятором.

Кластер с наиболее нетипичными параметрами, малой численностью или сильно отличающейся формой сигнала рассматривается как кластер аномальных или плохо классифицируемых сигналов.

### Перенумерация кластеров

Функция relabel_clusters перенумеровывает кластеры в более интерпретируемом порядке.

Логика:
1. Сначала ищем потенциально аномальный кластер:
    - самый маленький по размеру,
    - или кластер с наиболее нетипичным peak_idx,
    - или кластер с сильно отличающейся формой.

2. Оставшиеся два кластера сортируем по ratio_tail:
    - меньшая хвостовая компонента -> cluster 0,
    - большая хвостовая компонента -> cluster 1.

3. Аномальный кластер -> cluster 2.

In [ ]:
def relabel_clusters(labels, features_df):

    temp = features_df.copy()
    temp["cluster_raw"] = labels

    cluster_stats = temp.groupby("cluster_raw").agg(
        count=("max_amp", "size"),
        median_tail_ratio=("ratio_tail", "median"),
        median_peak_idx=("peak_idx", "median"),
        median_area=("area_total", "median"),
        median_amp=("max_amp", "median")
    )

    # Считаем, что аномальный кластер часто самый маленький
    anomaly_cluster = cluster_stats["count"].idxmin()

    normal_clusters = [c for c in cluster_stats.index if c != anomaly_cluster]

    # Два основных кластера сортируем по хвостовой компоненте
    normal_sorted = sorted(
        normal_clusters,
        key=lambda c: cluster_stats.loc[c, "median_tail_ratio"]
    )

    mapping = {
        normal_sorted[0]: 0,
        normal_sorted[1]: 1,
        anomaly_cluster: 2
    }

    relabeled = pd.Series(labels).map(mapping).values

    return relabeled, mapping, cluster_stats


best_labels, mapping, cluster_stats = relabel_clusters(best_labels_raw, features)

print("Mapping raw -> final:", mapping)
print()
print("Статистика сырых кластеров:")
display(cluster_stats)

print()
print("Размеры финальных кластеров:")
print(pd.Series(best_labels).value_counts().sort_index())

### Визуализация после перенумерации

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=best_labels,
    s=8,
    alpha=0.7
)
plt.title("Финальные кластеры после перенумерации")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="final cluster")
plt.show()

### Финальная средняя форма по кластерам

In [ ]:
plt.figure(figsize=(14, 6))

for cluster in sorted(np.unique(best_labels)):
    idx = np.where(best_labels == cluster)[0]
    mean_signal = processed_signals.iloc[idx].mean(axis=0)
    plt.plot(mean_signal.values, label=f"cluster {cluster}, n={len(idx)}")

plt.title("Средняя форма сигнала по финальным кластерам")
plt.xlabel("Время, отсчёты")
plt.ylabel("Преобразованная амплитуда")
plt.legend()
plt.show()

In [ ]:
submission = pd.DataFrame({
    "index": np.arange(len(best_labels)),
    "cluster": best_labels.astype(int)
})

submission.head()

In [ ]:
SUBMISSION_PATH = os.path.join(PROJECT_DIR, "submission.csv") # Сохранение submission.csv на Google Drive

submission.to_csv(SUBMISSION_PATH, index=False)

print("Файл сохранён:", SUBMISSION_PATH)
print("Размер submission:", submission.shape)

submission.head()

In [ ]:
check = pd.read_csv(SUBMISSION_PATH)

print(check.shape)
print(check.head())
print(check.dtypes)
print()
print("Уникальные кластеры:", sorted(check["cluster"].unique()))
print("Количество объектов:", len(check))

## Все варианты перенумерации кластеров

In [ ]:
import itertools

permutation_dir = os.path.join(PROJECT_DIR, "submissions_permutations")
os.makedirs(permutation_dir, exist_ok=True)

unique_clusters = [0, 1, 2]

permutation_files = []

for perm in itertools.permutations(unique_clusters):
    perm_mapping = {
        0: perm[0],
        1: perm[1],
        2: perm[2]
    }

    perm_labels = pd.Series(best_labels).map(perm_mapping).values

    perm_submission = pd.DataFrame({
        "index": np.arange(len(perm_labels)),
        "cluster": perm_labels.astype(int)
    })

    file_name = f"submission_perm_{perm[0]}_{perm[1]}_{perm[2]}.csv"
    file_path = os.path.join(permutation_dir, file_name)

    perm_submission.to_csv(file_path, index=False)
    permutation_files.append(file_path)

print("Сохранены варианты:")
for file_path in permutation_files:
    print(file_path)

##### Пояснение

Перебор перестановок помог исключить ситуацию, когда низкий score связан только с неправильными номерами кластеров. Даже после проверки разных соответствий 0, 1, 2 качество оставалось недостаточным, поэтому потребовалось менять сами признаки и способ кластеризации.

## Сравнение моделей

В проекте были протестированы несколько моделей кластеризации:

- KMeans;
- MiniBatchKMeans;
- GaussianMixture.

Для сравнения использовались внутренние метрики качества кластеризации:

- Silhouette Score;
- Calinski-Harabasz Score;
- Davies-Bouldin Score.

In [ ]:
scores_df_sorted

## Вариант 1. Отдельная проверка KMeans

После первоначального автоматического выбора модели, KMeans был проверен отдельно, с разными вариантами перенумерации кластеров.

Это была одна из первых проверок на Kaggle, но внешняя оценка оказалась заметно хуже ожидаемой.

In [ ]:
best_model_name = "KMeans"
best_labels_raw = labels_dict[best_model_name]

print("Выбранная модель:", best_model_name)
print("Размеры кластеров:")
print(pd.Series(best_labels_raw).value_counts().sort_index())

In [ ]:
cluster_analysis = features.copy()
cluster_analysis["cluster_raw"] = best_labels_raw

summary_kmeans = cluster_analysis.groupby("cluster_raw")[[
    "max_amp",
    "area_total",
    "ratio_peak",
    "ratio_early",
    "ratio_middle",
    "ratio_tail",
    "tail_to_peak",
    "tail_to_early",
    "peak_idx",
    "width_half_max",
    "skew",
    "kurtosis"
]].median()

summary_kmeans["count"] = cluster_analysis.groupby("cluster_raw").size()

summary_kmeans

In [ ]:
plt.figure(figsize=(14, 6))

for cluster in sorted(np.unique(best_labels_raw)):
    idx = np.where(best_labels_raw == cluster)[0]
    mean_signal = processed_signals.iloc[idx].mean(axis=0)
    plt.plot(mean_signal.values, label=f"cluster {cluster}, n={len(idx)}")

plt.title("Средняя форма сигнала по кластерам KMeans")
plt.xlabel("Время, отсчёты")
plt.ylabel("Преобразованная амплитуда")
plt.legend()
plt.show()

In [ ]:
import itertools

kmeans_perm_dir = os.path.join(PROJECT_DIR, "kmeans_submissions_permutations")
os.makedirs(kmeans_perm_dir, exist_ok=True)

unique_clusters = [0, 1, 2]
kmeans_permutation_files = []

for perm in itertools.permutations(unique_clusters):
    mapping = {
        0: perm[0],
        1: perm[1],
        2: perm[2]
    }

    perm_labels = pd.Series(best_labels_raw).map(mapping).values

    perm_submission = pd.DataFrame({
        "index": np.arange(len(perm_labels)),
        "cluster": perm_labels.astype(int)
    })

    file_name = f"kmeans_submission_perm_{perm[0]}_{perm[1]}_{perm[2]}.csv"
    file_path = os.path.join(kmeans_perm_dir, file_name)

    perm_submission.to_csv(file_path, index=False)
    kmeans_permutation_files.append(file_path)

print("Сохранены варианты KMeans:")
for file_path in kmeans_permutation_files:
    print(file_path)

## Вариант 2. Признаки формы и хвостовой части импульса

Первая группа моделей в основном использовала общие статистические признаки. После невысокого результата была другая идея: сильнее учитывать форму импульса и его хвостовую часть.

Для этого сигнал нормализовался относительно baseline, а затем рассчитывались интегралы по нескольким временным окнам, отношения tail/total, характеристики накопленной площади, ширины и производные.

In [ ]:
# Нормализация формы сигналов:
# 1) вычитаем базовую линию,
# 2) оставляем положительную часть,
# 3) нормируем отдельно по максимуму и по площади

X_raw = signals.values.astype(float)

# Базовая линия: медиана первых 100 отсчётов
baseline = np.median(X_raw[:, :100], axis=1, keepdims=True)

# Переворачиваем сигнал так же, как раньше, но адаптивно по baseline
X_pulse = baseline - X_raw

# Убираем отрицательные значения
X_pulse = np.clip(X_pulse, 0, None)

# Нормировка по максимуму
X_norm_max = X_pulse / (X_pulse.max(axis=1, keepdims=True) + 1e-9)

# Нормировка по площади
X_norm_area = X_pulse / (X_pulse.sum(axis=1, keepdims=True) + 1e-9)

print("X_pulse:", X_pulse.shape)
print("X_norm_max:", X_norm_max.shape)
print("X_norm_area:", X_norm_area.shape)

In [ ]:
plt.figure(figsize=(16, 6))
plt.plot(X_norm_max[:200].T, alpha=0.15)
plt.title("Нормализованные по максимуму сигналы")
plt.xlabel("Время, отсчёты")
plt.ylabel("Нормированная амплитуда")
plt.show()

In [ ]:
def extract_psd_features(X_pulse):
    """
    Извлекает признаки, связанные с формой сцинтилляционного импульса:
    интегралы по окнам, tail/total, short/long, моменты времени и спад.
    """

    X = X_pulse.astype(float)
    n_objects, n_points = X.shape

    features = pd.DataFrame(index=np.arange(n_objects))

    total = X.sum(axis=1) + 1e-9
    peak_idx = X.argmax(axis=1)
    max_amp = X.max(axis=1) + 1e-9

    features["max_amp"] = max_amp
    features["total_area"] = total
    features["peak_idx"] = peak_idx

    # Фиксированные окна вокруг импульса
    windows = {
        "pre": (0, 120),
        "rise": (120, 145),
        "peak": (145, 175),
        "short": (130, 190),
        "medium": (190, 280),
        "tail1": (190, 350),
        "tail2": (280, n_points),
        "long": (130, n_points)
    }

    for name, (start, end) in windows.items():
        start = max(0, start)
        end = min(n_points, end)
        area = X[:, start:end].sum(axis=1)
        features[f"area_{name}"] = area
        features[f"ratio_{name}"] = area / total

    # PSD-признаки
    features["tail1_total"] = features["area_tail1"] / total
    features["tail2_total"] = features["area_tail2"] / total
    features["tail1_short"] = features["area_tail1"] / (features["area_short"] + 1e-9)
    features["tail2_short"] = features["area_tail2"] / (features["area_short"] + 1e-9)
    features["short_long"] = features["area_short"] / (features["area_long"] + 1e-9)

    # Центр масс сигнала
    t = np.arange(n_points)
    features["center_of_mass"] = (X * t).sum(axis=1) / total

    # Времена накопления 10%, 50%, 90% площади
    cumsum = np.cumsum(X, axis=1)
    norm_cumsum = cumsum / total[:, None]

    for q in [0.1, 0.25, 0.5, 0.75, 0.9]:
        features[f"t_area_{int(q*100)}"] = (norm_cumsum >= q).argmax(axis=1)

    features["rise_area_10_50"] = features["t_area_50"] - features["t_area_10"]
    features["fall_area_50_90"] = features["t_area_90"] - features["t_area_50"]
    features["width_area_10_90"] = features["t_area_90"] - features["t_area_10"]

    # Ширины на разных уровнях максимума
    for level in [0.1, 0.25, 0.5, 0.75]:
        widths = []

        for i in range(n_objects):
            above = np.where(X[i] >= max_amp[i] * level)[0]

            if len(above) > 0:
                widths.append(above[-1] - above[0])
            else:
                widths.append(0)

        features[f"width_{int(level*100)}"] = widths

    # Статистики формы по нормированному сигналу
    X_norm = X / max_amp[:, None]

    features["norm_mean"] = X_norm.mean(axis=1)
    features["norm_std"] = X_norm.std(axis=1)
    features["norm_skew"] = stats.skew(X_norm, axis=1)
    features["norm_kurtosis"] = stats.kurtosis(X_norm, axis=1)

    # Производные
    dX = np.diff(X_norm, axis=1)
    features["diff_max"] = dX.max(axis=1)
    features["diff_min"] = dX.min(axis=1)
    features["diff_std"] = dX.std(axis=1)

    # Логарифмы масштабных признаков
    for col in ["max_amp", "total_area", "area_short", "area_medium", "area_tail1", "area_tail2"]:
        features[f"log_{col}"] = np.log1p(features[col])

    features = features.replace([np.inf, -np.inf], np.nan)
    features = features.fillna(features.median(numeric_only=True))

    return features

In [ ]:
features_psd = extract_psd_features(X_pulse)

print("Размер новых признаков:", features_psd.shape)
features_psd.head()

In [ ]:
selected_psd_features = [
    "log_max_amp",
    "log_total_area",
    "log_area_short",
    "log_area_medium",
    "log_area_tail1",
    "log_area_tail2",
    "ratio_short",
    "ratio_medium",
    "ratio_tail1",
    "ratio_tail2",
    "tail1_total",
    "tail2_total",
    "tail1_short",
    "tail2_short",
    "short_long",
    "center_of_mass",
    "t_area_10",
    "t_area_25",
    "t_area_50",
    "t_area_75",
    "t_area_90",
    "rise_area_10_50",
    "fall_area_50_90",
    "width_area_10_90",
    "width_10",
    "width_25",
    "width_50",
    "width_75",
    "norm_mean",
    "norm_std",
    "norm_skew",
    "norm_kurtosis",
    "diff_max",
    "diff_min",
    "diff_std"
]

X2 = features_psd[selected_psd_features].copy()

scaler2 = RobustScaler()
X2_scaled = scaler2.fit_transform(X2)

pca2 = PCA(n_components=12, random_state=RANDOM_STATE)
X2_pca = pca2.fit_transform(X2_scaled)

print("Размер X2_scaled:", X2_scaled.shape)
print("Размер X2_pca:", X2_pca.shape)
print("Суммарная объяснённая дисперсия PCA:", pca2.explained_variance_ratio_.sum())

In [ ]:
X2_model = X2_pca[:, :8]

models2 = {
    "KMeans_PSD": KMeans(
        n_clusters=3,
        n_init=100,
        random_state=RANDOM_STATE
    ),
    "MiniBatchKMeans_PSD": MiniBatchKMeans(
        n_clusters=3,
        batch_size=2048,
        n_init=100,
        random_state=RANDOM_STATE
    ),
    "GaussianMixture_PSD": GaussianMixture(
        n_components=3,
        covariance_type="full",
        n_init=20,
        random_state=RANDOM_STATE
    )
}

labels_dict2 = {}
scores2 = []

for name, model in models2.items():
    labels = model.fit_predict(X2_model)
    labels_dict2[name] = labels
    scores2.append(evaluate_clustering(X2_model, labels, name))

scores2_df = pd.DataFrame(scores2).sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True]
)

scores2_df

In [ ]:
best_psd_model_name = "KMeans_PSD"
best_psd_labels_raw = labels_dict2[best_psd_model_name]

print("Выбранная PSD-модель:", best_psd_model_name)
print(pd.Series(best_psd_labels_raw).value_counts().sort_index())

In [ ]:
cluster_analysis_psd = features_psd.copy()
cluster_analysis_psd["cluster_raw"] = best_psd_labels_raw

summary_psd = cluster_analysis_psd.groupby("cluster_raw")[[
    "max_amp",
    "total_area",
    "ratio_short",
    "ratio_medium",
    "ratio_tail1",
    "ratio_tail2",
    "tail1_total",
    "tail2_total",
    "tail1_short",
    "tail2_short",
    "short_long",
    "center_of_mass",
    "width_area_10_90",
    "norm_skew",
    "norm_kurtosis"
]].median()

summary_psd["count"] = cluster_analysis_psd.groupby("cluster_raw").size()
summary_psd

In [ ]:
psd_perm_dir = os.path.join(PROJECT_DIR, "psd_kmeans_submissions_permutations")
os.makedirs(psd_perm_dir, exist_ok=True)

psd_permutation_files = []

for perm in itertools.permutations([0, 1, 2]):
    mapping = {
        0: perm[0],
        1: perm[1],
        2: perm[2]
    }

    perm_labels = pd.Series(best_psd_labels_raw).map(mapping).values

    perm_submission = pd.DataFrame({
        "index": np.arange(len(perm_labels)),
        "cluster": perm_labels.astype(int)
    })

    file_name = f"psd_kmeans_submission_perm_{perm[0]}_{perm[1]}_{perm[2]}.csv"
    file_path = os.path.join(psd_perm_dir, file_name)

    perm_submission.to_csv(file_path, index=False)
    psd_permutation_files.append(file_path)

print("Сохранены PSD-варианты:")
for file_path in psd_permutation_files:
    print(file_path)

## Вариант 3. Перебор числа PCA-компонент и моделей

Рзультат одной выбранной конфигурации оказался нестабильным, поэтому потом перебиралось разное число PCA-компонент и несколько вариантов моделей.

Для каждого варианта дополнительно создавались разные перестановки номеров кластеров, поскольку в кластеризации номера 0, 1 и 2 сами по себе не имеют физического смысла.

In [ ]:
experiment_results = []
experiment_labels = {}

pca_components_list = [2, 3, 4, 5, 6, 8, 10, 12]

for n_comp in pca_components_list:
    X_exp = X2_pca[:, :n_comp]

    experiment_models = {
        f"KMeans_PSD_PCA{n_comp}": KMeans(
            n_clusters=3,
            n_init=100,
            random_state=RANDOM_STATE
        ),
        f"GMM_full_PSD_PCA{n_comp}": GaussianMixture(
            n_components=3,
            covariance_type="full",
            n_init=20,
            random_state=RANDOM_STATE
        ),
        f"GMM_tied_PSD_PCA{n_comp}": GaussianMixture(
            n_components=3,
            covariance_type="tied",
            n_init=20,
            random_state=RANDOM_STATE
        ),
        f"GMM_diag_PSD_PCA{n_comp}": GaussianMixture(
            n_components=3,
            covariance_type="diag",
            n_init=20,
            random_state=RANDOM_STATE
        )
    }

    for name, model in experiment_models.items():
        labels = model.fit_predict(X_exp)
        experiment_labels[name] = labels

        metrics = evaluate_clustering(X_exp, labels, name)
        metrics["n_pca_components"] = n_comp
        experiment_results.append(metrics)

experiment_results_df = pd.DataFrame(experiment_results).sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True]
)

experiment_results_df.head(20)

In [ ]:
top_model_names = experiment_results_df.head(10)["model"].tolist()

multi_model_perm_dir = os.path.join(PROJECT_DIR, "multi_model_submissions")
os.makedirs(multi_model_perm_dir, exist_ok=True)

created_files = []

for model_name in top_model_names:
    labels_raw = experiment_labels[model_name]

    for perm in itertools.permutations([0, 1, 2]):
        mapping = {
            0: perm[0],
            1: perm[1],
            2: perm[2]
        }

        perm_labels = pd.Series(labels_raw).map(mapping).values

        perm_submission = pd.DataFrame({
            "index": np.arange(len(perm_labels)),
            "cluster": perm_labels.astype(int)
        })

        safe_model_name = model_name.replace("/", "_").replace(" ", "_")
        file_name = f"{safe_model_name}_perm_{perm[0]}_{perm[1]}_{perm[2]}.csv"
        file_path = os.path.join(multi_model_perm_dir, file_name)

        perm_submission.to_csv(file_path, index=False)
        created_files.append(file_path)

print("Создано файлов:", len(created_files))
print("Папка:", multi_model_perm_dir)

for file_path in created_files[:20]:
    print(file_path)

## Промежуточные выводы по ЧЕРНОВИКУ

В ходе проверки вариантов было несколько направлений:

1. статистические и амплитудные признаки исходных импульсов,
2. StandardScaler, RobustScaler и PCA с разным числом компонент,
3. KMeans, MiniBatchKMeans и GaussianMixture,
4. разные варианты перенумерации кластеров,
5. признаки формы импульса и доли хвостовой компоненты,
6. фиксированная и адаптивная предобработка,
7. большие и компактные наборы PSD-признаков.

Лучший score на этом этапе не превышал 0.46854.

Вывод: обычная кластеризация большого набора признаков и выбор модели только по Silhouette/Calinski-Harabasz/Davies-Bouldin недостаточны. Следующая версия решения должна сильнее опираться на структуру формы импульса и отдельно проверять, действительно ли найденные кластеры соответствуют двум физическим типам сигналов.

In [ ]:
result_table = features.copy()
result_table["cluster_raw"] = best_labels_raw
result_table["cluster_final"] = best_labels

RESULT_TABLE_PATH = os.path.join(PROJECT_DIR, "features_with_clusters.csv")
result_table.to_csv(RESULT_TABLE_PATH, index=False)

print("Файл с признаками и кластерами сохранён:")
print(RESULT_TABLE_PATH)

In [ ]:
experiment_results_df.head(10)

## Вариант 4. Предобработка по примеру исходного ноутбука

В отдельном варианте была возвращена фиксированная формула преобразования сигнала из файла test-dataset.ipynb: 2**14 - signal - 1560.

После этого признаки и модели пересчитывались заново. Этот вариант был нужен, чтобы проверить, не ухудшает ли результат адаптивная оценка baseline.

In [ ]:
X_pulse_ref = (2**14 - signals.values.astype(float) - 1560)
X_pulse_ref = np.clip(X_pulse_ref, 0, None)

features_psd_ref = extract_psd_features(X_pulse_ref)

print("Размер признаков:", features_psd_ref.shape)
features_psd_ref.head()

In [ ]:
X_ref = features_psd_ref[selected_psd_features].copy()

scaler_ref = RobustScaler()
X_ref_scaled = scaler_ref.fit_transform(X_ref)

pca_ref = PCA(n_components=12, random_state=RANDOM_STATE)
X_ref_pca = pca_ref.fit_transform(X_ref_scaled)

print("Размер X_ref_scaled:", X_ref_scaled.shape)
print("Размер X_ref_pca:", X_ref_pca.shape)
print("Суммарная объяснённая дисперсия PCA:", pca_ref.explained_variance_ratio_.sum())

In [ ]:
ref_results = []
ref_labels = {}

pca_components_list = [2, 3, 4, 5, 6, 8, 10, 12]

for n_comp in pca_components_list:
    X_exp = X_ref_pca[:, :n_comp]

    ref_models = {
        f"KMeans_REF_PCA{n_comp}": KMeans(
            n_clusters=3,
            n_init=100,
            random_state=RANDOM_STATE
        ),
        f"GMM_full_REF_PCA{n_comp}": GaussianMixture(
            n_components=3,
            covariance_type="full",
            n_init=20,
            random_state=RANDOM_STATE
        ),
        f"GMM_tied_REF_PCA{n_comp}": GaussianMixture(
            n_components=3,
            covariance_type="tied",
            n_init=20,
            random_state=RANDOM_STATE
        ),
        f"GMM_diag_REF_PCA{n_comp}": GaussianMixture(
            n_components=3,
            covariance_type="diag",
            n_init=20,
            random_state=RANDOM_STATE
        )
    }

    for name, model in ref_models.items():
        labels = model.fit_predict(X_exp)
        ref_labels[name] = labels

        metrics = evaluate_clustering(X_exp, labels, name)
        metrics["n_pca_components"] = n_comp
        ref_results.append(metrics)

ref_results_df = pd.DataFrame(ref_results).sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True]
)

ref_results_df.head(20)

In [ ]:
top_ref_model_names = ref_results_df.head(10)["model"].tolist()

ref_perm_dir = os.path.join(PROJECT_DIR, "ref_model_submissions")
os.makedirs(ref_perm_dir, exist_ok=True)

ref_created_files = []

for model_name in top_ref_model_names:
    labels_raw = ref_labels[model_name]

    for perm in itertools.permutations([0, 1, 2]):
        mapping = {
            0: perm[0],
            1: perm[1],
            2: perm[2]
        }

        perm_labels = pd.Series(labels_raw).map(mapping).values

        perm_submission = pd.DataFrame({
            "index": np.arange(len(perm_labels)),
            "cluster": perm_labels.astype(int)
        })

        file_name = f"{model_name}_perm_{perm[0]}_{perm[1]}_{perm[2]}.csv"
        file_path = os.path.join(ref_perm_dir, file_name)

        perm_submission.to_csv(file_path, index=False)
        ref_created_files.append(file_path)

print("Создано файлов:", len(ref_created_files))
print("Папка:", ref_perm_dir)

for file_path in ref_created_files[:20]:
    print(file_path)

In [ ]:
ref_results_df.head(10)

## Вариант 5. Компактный набор PSD-признаков

После большого набора признаков была проверена противоположная идея: оставить только несколько наиболее интерпретируемых характеристик - площадь сигнала, амплитуду, долю хвоста, центр масс и ширину.

Цель этого шага - проверить, не мешает ли модели избыточное количество коррелирующих признаков.

In [ ]:
# Используем обработку из примера test-dataset:
# X_pulse_ref = 2**14 - signal - 1560

compact_features = pd.DataFrame()

compact_features["log_total_area"] = np.log1p(features_psd_ref["total_area"])
compact_features["log_max_amp"] = np.log1p(features_psd_ref["max_amp"])
compact_features["tail1_total"] = features_psd_ref["tail1_total"]
compact_features["tail2_total"] = features_psd_ref["tail2_total"]
compact_features["short_long"] = features_psd_ref["short_long"]
compact_features["center_of_mass"] = features_psd_ref["center_of_mass"]
compact_features["width_area_10_90"] = features_psd_ref["width_area_10_90"]

compact_features.head()

In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(
    compact_features["log_total_area"],
    compact_features["tail1_total"],
    s=5,
    alpha=0.5
)
plt.title("PSD-пространство: log_total_area vs tail1_total")
plt.xlabel("log_total_area")
plt.ylabel("tail1_total")
plt.show()

plt.figure(figsize=(10, 7))
plt.scatter(
    compact_features["log_max_amp"],
    compact_features["tail1_total"],
    s=5,
    alpha=0.5
)
plt.title("PSD-пространство: log_max_amp vs tail1_total")
plt.xlabel("log_max_amp")
plt.ylabel("tail1_total")
plt.show()

In [ ]:
compact_feature_sets = {
    "area_tail": [
        "log_total_area",
        "tail1_total"
    ],
    "amp_tail": [
        "log_max_amp",
        "tail1_total"
    ],
    "area_amp_tail": [
        "log_total_area",
        "log_max_amp",
        "tail1_total"
    ],
    "area_tail_short": [
        "log_total_area",
        "tail1_total",
        "short_long"
    ],
    "area_tail_center": [
        "log_total_area",
        "tail1_total",
        "center_of_mass"
    ],
    "area_tail_width": [
        "log_total_area",
        "tail1_total",
        "width_area_10_90"
    ],
    "area_tail2": [
        "log_total_area",
        "tail2_total"
    ],
    "area_tail_tail2": [
        "log_total_area",
        "tail1_total",
        "tail2_total"
    ]
}

compact_results = []
compact_labels = {}

for set_name, cols in compact_feature_sets.items():
    X_comp = compact_features[cols].copy()

    scaler_comp = StandardScaler()
    X_comp_scaled = scaler_comp.fit_transform(X_comp)

    models_comp = {
        f"KMeans_COMPACT_{set_name}": KMeans(
            n_clusters=3,
            n_init=200,
            random_state=RANDOM_STATE
        ),
        f"GMM_full_COMPACT_{set_name}": GaussianMixture(
            n_components=3,
            covariance_type="full",
            n_init=50,
            random_state=RANDOM_STATE
        ),
        f"GMM_tied_COMPACT_{set_name}": GaussianMixture(
            n_components=3,
            covariance_type="tied",
            n_init=50,
            random_state=RANDOM_STATE
        ),
        f"GMM_diag_COMPACT_{set_name}": GaussianMixture(
            n_components=3,
            covariance_type="diag",
            n_init=50,
            random_state=RANDOM_STATE
        )
    }

    for model_name, model in models_comp.items():
        labels = model.fit_predict(X_comp_scaled)

        compact_labels[model_name] = labels

        metrics = evaluate_clustering(X_comp_scaled, labels, model_name)
        metrics["features"] = ", ".join(cols)
        compact_results.append(metrics)

compact_results_df = pd.DataFrame(compact_results).sort_values(
    by=["silhouette", "calinski_harabasz", "davies_bouldin"],
    ascending=[False, False, True]
)

compact_results_df.head(20)

In [ ]:
top_compact_model_names = compact_results_df.head(10)["model"].tolist()

compact_perm_dir = os.path.join(PROJECT_DIR, "compact_model_submissions")
os.makedirs(compact_perm_dir, exist_ok=True)

compact_created_files = []

for model_name in top_compact_model_names:
    labels_raw = compact_labels[model_name]

    for perm in itertools.permutations([0, 1, 2]):
        mapping = {
            0: perm[0],
            1: perm[1],
            2: perm[2]
        }

        perm_labels = pd.Series(labels_raw).map(mapping).values

        perm_submission = pd.DataFrame({
            "index": np.arange(len(perm_labels)),
            "cluster": perm_labels.astype(int)
        })

        file_name = f"{model_name}_perm_{perm[0]}_{perm[1]}_{perm[2]}.csv"
        file_path = os.path.join(compact_perm_dir, file_name)

        perm_submission.to_csv(file_path, index=False)
        compact_created_files.append(file_path)

print("Создано файлов:", len(compact_created_files))
print("Папка:", compact_perm_dir)

for file_path in compact_created_files[:20]:
    print(file_path)

In [ ]:
compact_results_df.head(10)